# Notebook 04: Negative Control Evaluation

Validating that the TransE baseline model actually discriminates — i.e., drugs with no plausible mechanistic relationship to brain injury should score significantly lower than the top candidates. This is a standard sanity check in drug repurposing pipelines.

## Setup

In [1]:
import numpy as np
import pandas as pd
import csv
import torch
import torch.nn.functional as fn
import sys
sys.path.insert(1, '../utils')
from utils import download_and_extract

download_and_extract()

## Define negative control drugs

These are FDA-approved drugs with no established or plausible relationship to acute brain injury — selected from disease areas with no known neurological overlap.

In [8]:
# Negative controls: drugs with no plausible brain injury mechanism
# Selected from dermatology, antifungals, and GI indications
negative_controls = {
    'Compound::DB00951': 'Isoniazid',        # TB antibiotic
    'Compound::DB00635': 'Prednisone',        # corticosteroid - actually has some brain relevance, swap if needed
    'Compound::DB01167': 'Itraconazole',      # antifungal
    'Compound::DB00688': 'Mycophenolate',     # immunosuppressant for transplant
    'Compound::DB00502': 'Haloperidol',       # antipsychotic
    'Compound::DB00682': 'Warfarin',          # known positive control - stroke prevention
    'Compound::DB00945': 'Aspirin',           # known positive control - stroke prevention
}

# separate true negatives from positive controls we included for comparison
true_negatives = {
    'Compound::DB00951': 'Isoniazid',
    'Compound::DB01167': 'Itraconazole',
    'Compound::DB00688': 'Mycophenolate',
}

known_positives = {
    'Compound::DB00682': 'Warfarin',
    'Compound::DB00945': 'Aspirin',
}


## Load embeddings and maps

In [9]:
entity_map = {}
entity_id_map = {}
relation_map = {}

with open('../data/embed/entities.tsv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t', fieldnames=['name', 'id']):
        entity_map[row['name']] = int(row['id'])
        entity_id_map[int(row['id'])] = row['name']

with open('../data/embed/relations.tsv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t', fieldnames=['name', 'id']):
        relation_map[row['name']] = int(row['id'])

entity_emb = np.load('../data/embed/DRKG_TransE_l2_entity.npy')
rel_emb = np.load('../data/embed/DRKG_TransE_l2_relation.npy')

## Score negative controls and known positives

In [10]:
brain_injury_disease_list = [
    'Disease::MESH:D020521',
    'Disease::MESH:D002544',
    'Disease::MESH:D020300',
    'Disease::MESH:D020520',
    'Disease::MESH:D002538',
    'Disease::MESH:D001930',
    'Disease::MESH:D006470',
]

treatment_relations = [
    'Hetionet::CtD::Compound:Disease',
    'GNBR::T::Compound:Disease'
]

gamma = 12.0

def transE_l2(head, rel, tail):
    return gamma - torch.norm(head + rel - tail, p=2, dim=-1)

disease_ids = [entity_map[d] for d in brain_injury_disease_list if d in entity_map]
treatment_rid = [relation_map[r] for r in treatment_relations if r in relation_map]
treatment_embs = [torch.tensor(rel_emb[rid]) for rid in treatment_rid]

def score_drug(drugbank_id):
    if drugbank_id not in entity_map:
        return None
    drug_emb = torch.tensor(entity_emb[entity_map[drugbank_id]]).unsqueeze(0)
    scores = []
    for treatment_emb in treatment_embs:
        for disease_id in disease_ids:
            disease_emb = torch.tensor(entity_emb[disease_id])
            s = fn.logsigmoid(transE_l2(drug_emb, treatment_emb, disease_emb))
            scores.append(s.item())
    return np.mean(scores)

## Results

In [11]:
results = []

for drug_id, name in {**true_negatives, **known_positives}.items():
    score = score_drug(drug_id)
    category = 'negative control' if drug_id in true_negatives else 'known positive'
    results.append({
        'drug': name,
        'drugbank_id': drug_id,
        'category': category,
        'mean_score': score
    })

results_df = pd.DataFrame(results).sort_values('mean_score', ascending=False)
results_df

,drug,drugbank_id,category,mean_score
4,Aspirin,Compound::DB00945,known positive,-1.543750
3,Warfarin,Compound::DB00682,known positive,-1.760804
2,Mycophenolate,Compound::DB00688,negative control,-2.059639
1,Itraconazole,Compound::DB01167,negative control,-2.295571
0,Isoniazid,Compound::DB00951,negative control,-2.315834


## Compare against top candidate scores

How do negative controls compare to the top candidates from Notebook 02?

In [12]:
top100 = pd.read_csv('../results/transE_baseline_top100.csv')

print(f"Top candidate score range: {top100['score'].max():.4f} to {top100['score'].min():.4f}")
print(f"Top candidate mean score: {top100['score'].mean():.4f}")
print(f"\nNegative control scores:")
for _, row in results_df[results_df['category'] == 'negative control'].iterrows():
    print(f"  {row['drug']}: {row['mean_score']:.4f}")
print(f"\nKnown positive scores:")
for _, row in results_df[results_df['category'] == 'known positive'].iterrows():
    print(f"  {row['drug']}: {row['mean_score']:.4f}")

Top candidate score range: -0.0752 to -0.2220
Top candidate mean score: -0.1779

Negative control scores:
  Mycophenolate: -2.0596
  Itraconazole: -2.2956
  Isoniazid: -2.3158

Known positive scores:
  Aspirin: -1.5438
  Warfarin: -1.7608


## Save

In [13]:
import os
os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/negative_controls.csv', index=False)